Sheet3 (ground truth mốc năm)

↓

Merge EXACT theo year

↓

Tạo missing cho năm giữa

↓

Interpolate

In [71]:
import pandas as pd
import numpy as np

### Đọc file & chuẩn hóa tên cột Sheet Tổng hợp

In [72]:
all_sheets = pd.read_excel('../data/interim/data-off.xlsx', sheet_name=None)

# Duyệt qua từng sheet
for sheet_name, df in all_sheets.items():
    print(f"Sheet: {sheet_name}")
    print(df.head())
    print()

Sheet: Mẫu của cô ( tham khảo )
             Country             Unnamed: 1    Year         EV   GDPexporter  \
0  Australia_Belgium  Australia_Belgium2010  2010.0  1158189.0  1.147589e+12   
1  Australia_Belgium  Australia_Belgium2011  2011.0  1471194.0  1.397908e+12   
2  Australia_Belgium  Australia_Belgium2012  2012.0  1433890.0  1.546509e+12   
3  Australia_Belgium  Australia_Belgium2013  2013.0  1036383.0  1.576335e+12   
4  Australia_Belgium  Australia_Belgium2014  2014.0   912551.0  1.467505e+12   

    GDPimporter  POPexporter  POPimporter     DIST  FTA1  ...  LANG  LANDLOCK  \
0  4.814209e+11   22031750.0   10895586.0  16277.0   0.0  ...   0.0       1.0   
1  5.233304e+11   22340024.0   11038264.0  16277.0   0.0  ...   0.0       1.0   
2  4.961529e+11   22733465.0   11106932.0  16277.0   0.0  ...   0.0       1.0   
3  5.217910e+11   23128129.0   11159407.0  16277.0   0.0  ...   0.0       1.0   
4  5.353902e+11   23475686.0   11209057.0  16277.0   0.0  ...   0.0       1.0   



### Chuẩn hóa tên cột

In [73]:
sheet2 = all_sheets['Tổng hợp'].copy()

# Lấy dòng đầu làm header
sheet2.columns = sheet2.iloc[0]
sheet2 = sheet2[1:].reset_index(drop=True)

# Replace NaN column names
sheet2.columns = [
    f"col_{i}" if pd.isna(col) else str(col).strip()
    for i, col in enumerate(sheet2.columns)
]

print(sheet2.columns.tolist())
sheet2.head()

['iso_o', 'iso_d', 'col_2', 'year', 'col_4', 'col_5', 'Di Cư', 'GDP NƯỚC ĐI', 'GDP NƯỚC ĐẾN', 'GDP BQĐN NƯỚC ĐI', 'GDP BQĐN NƯỚC ĐẾN', 'inflation_cpi_org ( chỉ số lạm phát )', 'inflation_cpi_des', 'dist (Khoảng cách địa lý)', 'distcap (Khoảng cách thủ đô)', 'distwces (khoảng cách kinh tế)', 'comlang_off (Ngôn ngữ chung )', 'contig ( biên giới chung )', 'colony ( Thuộc địa)', 'pop_total_org ( tổng dân số nước đi )', 'pop_total_des (dân số nước đến)', 'disaster_count-org ( tỉ số thiên tai )', 'disaster_count-des ( tỉ số thiên tai nước đến )', 'co2_org ( nồng độ co2 bq đầu người )', 'co2_des', 'pm25_exceed_org(tỷ lệ dân tiếp xúc bụi)', 'pm25_exceed_des', 'pm25_org( nồng độ bụi trung bình )', 'pm25_des', 'internet_use_org ( cá nhân dùng internet % dân số )', 'internet_use_des', 'cá nhân sd internet_org', 'cá nhân sd internet_des', 'Băng thông Internet quốc tế bình quân đầu người dùng_org', 'Băng thông Internet quốc tế bình quân đầu người dùng_des']


,iso_o,iso_d,col_2,year,col_4,col_5,Di Cư,GDP NƯỚC ĐI,GDP NƯỚC ĐẾN,GDP BQĐN NƯỚC ĐI,...,pm25_exceed_org(tỷ lệ dân tiếp xúc bụi),pm25_exceed_des,pm25_org( nồng độ bụi trung bình ),pm25_des,internet_use_org ( cá nhân dùng internet % dân số ),internet_use_des,cá nhân sd internet_org,cá nhân sd internet_des,Băng thông Internet quốc tế bình quân đầu người dùng_org,Băng thông Internet quốc tế bình quân đầu người dùng_des
0,AUS,BEL,AUS_BEL,2000,AUS2000,BEL2000,NaN,416167815092.90802,236792460312.471008,26541.666016,...,33.832069,99.275414,7.457433,19.489698,46.7561,29.4317,0.467561,0.294317,274.381012,6215.919922
1,AUS,BEL,AUS_BEL,2001,AUS2001,BEL2001,NaN,379629301675.107971,236746141604.369995,27645.814453,...,0,0,7.388954,19.077738,52.689301,31.288401,0.526893,0.312884,688.479004,25298.400391
2,AUS,BEL,AUS_BEL,2002,AUS2002,BEL2002,NaN,395788696012.059021,258383599375.177002,29032.490234,...,0,0,7.266338,18.59227,0,46.330002,0,0.4633,0,17551.6
3,AUS,BEL,AUS_BEL,2003,AUS2003,BEL2003,NaN,467739079790.33197,318082528506.588013,30121.818359,...,0,0,7.131202,18.1085,0,49.970001,0,0.4997,0,16219.6
4,AUS,BEL,AUS_BEL,2004,AUS2004,BEL2004,NaN,614659980082.515015,369214712443.205994,31763.796875,...,0,0,7.025164,17.701635,0,53.860001,0,0.5386,0,20940.1


In [74]:
# ── Chuẩn hóa tên cột ──
sheet2.rename(columns={
    'col_2': 'pair_id',
    'col_4': 'iso_o_year',
    'col_5': 'iso_d_year'
}, inplace=True)
rename_map = {
    'Di Cư': 'migration',
    'GDP NƯỚC ĐI': 'gdp_org',
    'GDP NƯỚC ĐẾN': 'gdp_des',
    'GDP BQĐN NƯỚC ĐI': 'gdppc_org',
    'GDP BQĐN NƯỚC ĐẾN': 'gdppc_des',
    'inflation_cpi_org ( chỉ số lạm phát )': 'inflation_org',
    'inflation_cpi_des': 'inflation_des',
    'dist (Khoảng cách địa lý)': 'dist',
    'distcap (Khoảng cách thủ đô)': 'distcap',
    'distwces (khoảng cách kinh tế)': 'distwces',
    'comlang_off (Ngôn ngữ chung )': 'comlang_off',
    'contig ( biên giới chung )': 'contig',
    'colony ( Thuộc địa)': 'colony',
    'pop_total_org ( tổng dân số nước đi )': 'pop_org',
    'pop_total_des (dân số nước đến)': 'pop_des',
    'disaster_count-org ( tỉ số thiên tai )': 'disaster_org',
    'disaster_count-des ( tỉ số thiên tai nước đến )': 'disaster_des',
    'co2_org ( nồng độ co2 bq đầu người )': 'co2_org',
    'co2_des': 'co2_des',  # đã strip ở bước trước
    'pm25_exceed_org(tỷ lệ dân tiếp xúc bụi)': 'pm25_exceed_org',
    'pm25_exceed_des': 'pm25_exceed_des',
    'pm25_org( nồng độ bụi trung bình )': 'pm25_org',
    'pm25_des': 'pm25_des',
    'internet_use_org ( cá nhân dùng internet % dân số )': 'internet_use_org',
    'internet_use_des': 'internet_use_des',
    'cá nhân sd internet_org': 'internet_indiv_org',
    'cá nhân sd internet_des': 'internet_indiv_des',
    'Băng thông Internet quốc tế bình quân đầu người dùng_org': 'bandwidth_org',
    'Băng thông Internet quốc tế bình quân đầu người dùng_des': 'bandwidth_des',
}

sheet2.rename(columns=rename_map, inplace=True)

sheet2['iso_o'] = sheet2['iso_o'].str.strip()
sheet2['iso_d'] = sheet2['iso_d'].str.strip()
sheet2['year'] = pd.to_numeric(sheet2['year'], errors='coerce')

sheet2.head()

,iso_o,iso_d,pair_id,year,iso_o_year,iso_d_year,migration,gdp_org,gdp_des,gdppc_org,...,pm25_exceed_org,pm25_exceed_des,pm25_org,pm25_des,internet_use_org,internet_use_des,internet_indiv_org,internet_indiv_des,bandwidth_org,bandwidth_des
0,AUS,BEL,AUS_BEL,2000,AUS2000,BEL2000,NaN,416167815092.90802,236792460312.471008,26541.666016,...,33.832069,99.275414,7.457433,19.489698,46.7561,29.4317,0.467561,0.294317,274.381012,6215.919922
1,AUS,BEL,AUS_BEL,2001,AUS2001,BEL2001,NaN,379629301675.107971,236746141604.369995,27645.814453,...,0,0,7.388954,19.077738,52.689301,31.288401,0.526893,0.312884,688.479004,25298.400391
2,AUS,BEL,AUS_BEL,2002,AUS2002,BEL2002,NaN,395788696012.059021,258383599375.177002,29032.490234,...,0,0,7.266338,18.59227,0,46.330002,0,0.4633,0,17551.6
3,AUS,BEL,AUS_BEL,2003,AUS2003,BEL2003,NaN,467739079790.33197,318082528506.588013,30121.818359,...,0,0,7.131202,18.1085,0,49.970001,0,0.4997,0,16219.6
4,AUS,BEL,AUS_BEL,2004,AUS2004,BEL2004,NaN,614659980082.515015,369214712443.205994,31763.796875,...,0,0,7.025164,17.701635,0,53.860001,0,0.5386,0,20940.1


### Đọc Sheet Di cư & fill vào cột migration (nearest year)

In [75]:
# Chuẩn hóa tên cột và lọc lấy các cột cần thiết
sheet3 = all_sheets['Di cư']
# Chuẩn hóa tên cột
sheet3.rename(columns={
    'DES'       : 'iso_d',
    'DES ( Đến )': 'country_des',
    'ORI ( Đi )': 'country_ori',
    'ORI'       : 'iso_o',
}, inplace=True)
sheet3.head()

KeyError: 'Di cư'

In [ ]:
year_cols = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024]

# ── Melt: wide → long ──
sheet3 = sheet3.melt(
    id_vars=['iso_o', 'iso_d'],
    value_vars=year_cols,
    var_name='year_ref',
    value_name='migration_ref'
)
sheet3['year_ref'] = sheet3['year_ref'].astype(int)
sheet3['iso_o']    = sheet3['iso_o'].str.strip()
sheet3['iso_d']    = sheet3['iso_d'].str.strip()
sheet3.head()

,iso_o,iso_d,year_ref,migration_ref
0,HKG,CHN,1990,110881
1,JPN,CHN,1990,33057
2,KOR,CHN,1990,60637
3,BGD,CHN,1990,327
4,KHM,CHN,1990,123


In [ ]:
sheet2['migration'] = np.nan

sheet2 = sheet2.merge(
    sheet3[['iso_o', 'iso_d', 'year_ref', 'migration_ref']],
    left_on=['iso_o', 'iso_d', 'year'],
    right_on=['iso_o', 'iso_d', 'year_ref'],
    how='left'
)

sheet2['migration'] = sheet2['migration_ref']
sheet2.drop(columns=['year_ref', 'migration_ref'], inplace=True)

print(f"Đã fill: {sheet2['migration'].notna().sum()} / {len(sheet2)} dòng")
sheet2[['iso_o', 'iso_d', 'year', 'migration']].dropna(subset=['migration']).head(5)

Đã fill: 2190 / 16275 dòng


,iso_o,iso_d,year,migration
50,AUS,CAN,2000,15831.0
55,AUS,CAN,2005,17893.0
60,AUS,CAN,2010,24105.0
65,AUS,CAN,2015,30029.0
70,AUS,CAN,2020,26000.0


In [ ]:
# Các cặp có trong sheet2 nhưng không có trong sheet3
missing_pairs = (
    sheet2[['iso_o', 'iso_d']]
    .drop_duplicates()
    .merge(
        sheet3[['iso_o', 'iso_d']].drop_duplicates(),
        on=['iso_o', 'iso_d'],
        how='left',
        indicator=True
    )
)

print("Cặp KHÔNG có trong sheet3:")
print(missing_pairs[missing_pairs['_merge'] == 'left_only'])

Cặp KHÔNG có trong sheet3:
    iso_o iso_d     _merge
0     AUS   BEL  left_only
1     AUS   BGD  left_only
3     AUS   CHE  left_only
5     AUS   CZE  left_only
6     AUS   DEU  left_only
..    ...   ...        ...
593   UKR   LAO  left_only
598   UKR   THA  left_only
600   UKR   VNM  left_only
618   USA   LAO  left_only
625   USA   VNM  left_only

[286 rows x 3 columns]


### Nội suy tuyến tính dùng log/exp

In [ ]:
def log_linear_interpolate(s):
    s = s.replace(0, np.nan)
    return np.exp(np.log(s).interpolate(method='linear'))

sheet2 = sheet2.sort_values(['iso_o', 'iso_d', 'year'])

sheet2['migration'] = (
    sheet2.groupby(['iso_o', 'iso_d'])['migration']
          .transform(log_linear_interpolate)
)

# Làm tròn về số nguyên (di cư là người, không có số lẻ)
sheet2['migration'] = sheet2['migration'].round(0)

print(f"Sau nội suy: {sheet2['migration'].notna().sum()} / {len(sheet2)} dòng có giá trị")
print(f"Vẫn còn NaN (cặp không có trong Sheet DI cư): {sheet2['migration'].isna().sum()}")

Sau nội suy: 9100 / 16275 dòng có giá trị
Vẫn còn NaN (cặp không có trong Sheet DI cư): 7175


In [ ]:
# ── Kiểm tra kết quả: xem 1 cặp ──
sample = sheet2[sheet2['iso_o'].eq('AUS') & sheet2['iso_d'].eq('CAN')][['year','migration']]
print("\nAUS → CAN:")
print(sample.to_string(index=False))


AUS → CAN:
 year  migration
 2000    15831.0
 2001    16223.0
 2002    16626.0
 2003    17038.0
 2004    17460.0
 2005    17893.0
 2006    18992.0
 2007    20158.0
 2008    21396.0
 2009    22710.0
 2010    24105.0
 2011    25188.0
 2012    26320.0
 2013    27502.0
 2014    28738.0
 2015    30029.0
 2016    29176.0
 2017    28347.0
 2018    27542.0
 2019    26760.0
 2020    26000.0
 2021    26192.0
 2022    26386.0
 2023    26581.0
 2024    26777.0


### Lưu trữ dữ liệu

In [ ]:
# Tạo folder nếu chưa có
sheet2.to_csv('../data/raw/data_raw.csv', index=False)
print("✅ Đã lưu file CSV vào ../data/raw/")

✅ Đã lưu file CSV vào ../data/raw/


In [ ]:

# ── 1. Lưu CSV (machine-friendly) ──
sheet1 = all_sheets['Mẫu của cô ( tham khảo )']
# ── 2. Lưu XLSX (human-friendly) ──
with pd.ExcelWriter('../data/raw/data_raw.xlsx', engine='openpyxl') as writer:
    sheet1.to_excel(writer, sheet_name='sample_reference', index=False)
    sheet2.to_excel(writer, sheet_name='bilateral_panel', index=False)
    sheet3.to_excel(writer, sheet_name='migration_flows', index=False)

print("✅ Đã lưu cả CSV và XLSX vào ../data/raw/")

✅ Đã lưu cả CSV và XLSX vào ../data/raw/
